![](docs/Banner.jpg)

# InSAR Time Series Analysis: MintPy + NISAR GUNW products

**Author:** Changyang Hu, Zhang Yunjun, Heresh Fattahi, August 3-7, 2026 [EarthScope InSAR Short Course (ISCE+)](https://www.earthscope.org/event/2026-technical-course-insar-processing-and-analysis-isce/).

The Miami INsar Timeseries software in PYthon (MintPy) is an open-source package for InSAR time-series analysis. MintPy currently starts from stacks of unwrapped interferograms (in either geo- or radar-coordinates) and estimates ground displacement time-series. MintPy is primarily consistent with stacks of interferograms processed with ISCE. However, the software also supports interfarograms processed with other InSAR processors such as GAMMA, GMTSAR, SNAP and ROI_PAC. 

MintPy is available on Github from the following page: https://github.com/insarlab/MintPy

References: The detailed algorithms implemented in MintPy can be found in the following paper: 

+ Yunjun, Z., Fattahi, H., Amelung, F. (2019), Small baseline InSAR time series analysis: Unwrapping error correction and noise reduction, _Computers & Geosciences, 133,_ 104331. [ [doi](https://doi.org/10.1016/j.cageo.2019.104331) \| [arxiv](https://doi.org/10.31223/osf.io/9sz6m) \| [data](https://doi.org/10.5281/zenodo.3464190) \| [notebook](https://github.com/geodesymiami/Yunjun_et_al-2019-MintPy) ]

# Table of Contents

+ [0. Initial setup](#0.-Initial-setup-of-the-notebook)
+ [1. General overview](#1.-General-overview-of-smallbaselineApp.py)
   - [1.1 Processing steps](#1.1-Processing-steps-of-smallbaselineApp.py)
   - [1.2 Configuring the processing parameters](#1.2-Configuring-the-processing-parameters)
+ [2. Time series analysis](#2.-Time-series-analysis---small-baseline-approach)

# 0. Initial setup of the notebook

The cell below performs the intial setup of the notebook and must be **run every time the notebook (re)starts**. It defines the processing location and check the example dataset.

In [ ]:
%matplotlib inline
import os
import matplotlib.pyplot as plt
import numpy as np
import shutil
from cartopy import crs as ccrs
from mintpy.utils import readfile, utils as ut, plot as pp
from mintpy.cli import view, tsview, plot_network, plot_transection
from mintpy.view import prep_slice, plot_slice
import utils
plt.rcParams.update({'font.size': 12})

# utils function
def write_config_file(out_file, CONFIG_TXT, mode='a'): 
    """Write configuration files for MintPy to process NISAR sample products"""
    if not os.path.isfile(out_file) or mode == 'w':
        with open(out_file, "w") as fid:
            fid.write(CONFIG_TXT)
        print('write configuration to file: {}'.format(out_file))
    else:
        with open(out_file, "a") as fid:
            fid.write("\n" + CONFIG_TXT)
        print('add the following to file: \n{}'.format(CONFIG_TXT))

# define and go to the work directory
work_dir = os.path.expanduser('~/data/MexicoCityNisarDT113/mintpy')
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)
print('Go to work directory:', work_dir)

# define the custom config file
config_file = os.path.join(work_dir, "MexicoCityNisarDT113.txt")

## TODO

Run the cell below to 1) download the staged stack of interferograms or 2) to download and prepare the stack of interferograms using ARIA-tools. We have pre-processed an example ARIA dataset on San Francisco Bay and uploaded it to AWS and [Zenodo](https://zenodo.org/record/6990323), one could also run ARIA-tools commands to download and pre-process themselves, by setting `use_staged_data` below.

In [ ]:
# # TODO

# # download/prepare the interferogram stack from ARIA products and load into mintpy
# # aws    - download pre-processed data from AWS S3 bucket [recommended, requires awscli]
# # zenodo - download pre-processed data from zenodo using wget
# # False  - download & pre-process from ARIA using ARIA-tools
# use_staged_data = 'aws'  #['aws' / 'zenodo' / False]

# if all(os.path.isfile(os.path.join(work_dir, 'inputs', x)) for x in ['ifgramStack.h5', 'geometryGeo.h5']):
#     print('ARIA products are already loaded into MintPy. Skip re-loading.')

# elif use_staged_data in ['aws', 'zenodo']:
#     # option 1: download the staged data from AWS or Zenodo
#     os.chdir(os.path.dirname(os.path.dirname(work_dir)))
#     tar_file = os.path.join(os.path.dirname(os.path.dirname(work_dir)), 'SanFranSenDT42.tar.gz')
#     if os.path.isfile(tar_file):
#         print('Staged ARIA product exists at: {}'.format(tar_file))
#     elif use_staged_data == 'aws':
#         !aws --region us-west-2 --no-sign-request s3 cp s3://asf-jupyter-data-west/unavco2022/SanFranSenDT42.tar.gz ./
#     elif use_staged_data == 'zenodo':
#         !wget https://zenodo.org/record/6990323/files/SanFranSenDT42.tar.gz ./

#     # decompress the tar file [it takes ~1.5 min]
#     print('decompressing the downloaded dataset...')
#     !pv SanFranSenDT42.tar.gz | tar -xz
#     os.chdir(work_dir)

# elif use_staged_data is False:
#     # option 2: download / prepare ARIA products using ARIA-tools, and load into MintPy
#     aria_stack_files = [os.path.join(work_dir, f'../stack/{x}Stack.vrt') for x in ['unwrap', 'coh', 'connComp']]
#     if all(os.path.isfile(x) for x in stack_files):
#         print('ARIA products already exists at: {}'.format(os.path.dirname(work_dir)))
#     else:
#         print("Using ARIA-tools to download and prepare the input data for MintPy")
#         os.chdir(os.path.dirname(work_dir))
#         !ariaDownload.py -b '37.25 38.1 -122.6 -121.75' --track 42
#         !ariaTSsetup.py -f 'products/*.nc' -b '37.25 38.1 -122.6 -121.75' --mask Download --num_threads 4 --verbose
#         os.chdir(work_dir)
#     # load ARIA products into MintPy
#     !prep_aria.py -s ../stack/ -d ../DEM/SRTM_3arcsec.dem -i ../incidenceAngle/*.vrt -a ../azimuthAngle/*.vrt -w ../mask/watermask.msk --update

#     # compress for staging
#     # tar cvzf SanFranSenDT42.tar.gz SanFranSenDT42
# else:
#     raise ValueError(f'un-recognized "use_staged_data" setting: {use_staged_data}\navailable settings: "aws", "zenodo", False.')

# 1. General overview of smallbaselineApp.py

This application provides a workflow which includes several steps to invert a stack of unwrapped interferograms and apply different corrections to obtain ground displacement timeseries. The workflow consists of two main blocks:

* correcting for unwrapping errors and inverting for the raw phase time-series (blue ovals)
* correcting for noise from different sources to obtain the displacement time-series (green ovals)

Some steps are optional, which are switched off by default (marked by dashed boundaries). Configuration parameters for each step are initiated with default values in a customizable text file: [smallbaselineApp.cfg](https://github.com/insarlab/MintPy/blob/main/src/mintpy/defaults/smallbaselineApp.cfg). In this notebook, we will walk through the various steps.

![](docs/smallbaselineApp_workflow.jpg)
<p style="text-align: center;">
    (Figure from Yunjun et al., 2019)
</p>

## 1.1 Processing steps of smallbaselineApp.py

The `smallbaselineApp.py` workflow can be called with a single command-line call. By default it will run all the required processing steps with options pulled from the template files, as shown in [section 5.2](#5.2-smallbaselineApp.py-non-stop-processing). However, in this notebook, we will use the "step" processing, which allows to re-start the processing from a given step. More detailed usage can be found in help.

In [ ]:
!smallbaselineApp.py --help

The app includes the following processing steps:

#### Input

* `load_data:` loads the stack of interferograms (unwrapped phase, coherence, and/or connected componenents) and geometry files (height, incidence/azimuth angle, lookup tables etc.) into HDF5 files with multiple datasets and attributes.  

#### Network inversion for time-series

* `modify_network:` this step (if requested) modifies the network of interferograms, e.g., based on average coherence, temporal and spatial baselines threshold, or by removing specific pairs.
* `reference_point:` the unwrapped interferograms may be relative to differnt reference pixels. This step introduces a common reference pixel to all interferograms. For intuitive interpretation, one may choose a stable coherent non-deforming pixel. However, since the estimated InSAR displacement time-series is relative in both time and space, choosing a deforming pixel does not change the results.
* `quick_overview:` this step provides a quick assessment of:
   1. expected rate maps even before inversion by simply averaging / stacking the unwrapped interferograms;
   2. distribution of unwrapping errors from the number of interferogram triplets with non-zero integer ambiguity.
* `correct_unwrap_error:` the input unwrapped interferograms may be affected by phase unwrapping errors (wrong integer number of $2\pi$ phase added during phase unwrapping). This step (if requested) offers three methods to possibly correct unwrapping errors.
* `invert_network:` inverts the stack of unwrapped interferograms to form the InSAR phase time-series. This is equivalent to transforming the network of small-baseline interferograms to a single-reference network of interferogram (i.e., the unwrapped phase timeseries). 

#### Noise reduction of displacement time-series

* `correct_LOD:` this step is specific to Envisat data and applies an empircal correction to account for possible local oscilator drift of the radar.
* `correct_SET:` corrects (if requested) the solid Earth tides due to the gravity pull from the Sun and the Moon.
* `correct_ionosphere`: corrects ionospheric delay using the split-spectrum results (from ISCE-2 stack processors only).
* `correct_troposphere:` corrects tropospheric delay 1) using atmospheric models or 2) with empirical phase elevation approach estimated from InSAR data.
* `deramp:` this step (if requested) removes a ramp from each acquisition. Note that deramping removes residual long-wavelength interferometric phase components which may be due to noise (geometrical residual, atmospheric delay) or signal (tectonic deformation).  
* `correct_topography:` estimates residual topographic effects (due to DEM errors) which are correlated with temporal variation of perpendicular baseline.  
* `residual_RMS:` estimates the average noise level for each acquisition by calculating the RMS of the residual phase.
* `reference_date:` change reference date.
* `velocity:` estimates a suite of time functions, such as a linear velocity.

#### Output

* `geocode:` if the original stack in radar-coordinates, convert it to geo-coordinates in lat/lon
* `google_earth:` output the average velocity into an Google Earth KMZ file.
* `hdfeos5:` output the displacement time-series with geometry info into one file in [HDF-EOS5](http://hdfeos.org) format.

## 1.2 Configuring the processing parameters

The processing parameters for the smallbaselineApp.py are controlled via configuration files. At least one file is required to run smallbaselineApp.py.

* `default configuration`: [smallbaselineApp.cfg](https://github.com/insarlab/MintPy/blob/main/src/mintpy/defaults/smallbaselineApp.cfg). It contains all configuration parameters, grouped by steps, with default auto values (which are defined in [smallbaselineApp_auto.cfg](https://github.com/insarlab/MintPy/blob/main/src/mintpy/defaults/smallbaselineApp_auto.cfg)). This file is copied over to the current working directory and read every time smallbaselineApp.py runs.
* `custom configuration` (optional but recommended): `MexicoCityNisarDT113.txt` in the example dataset. It constains selective, manually modified configuration parameters. The custom template file name is arbitrary. Custom template has higher priority than the default template; if custom template is specified, smallbaselineApp.py will update the default smallbaselineApp.cfg file accordingly.

### Custom configuration for the dataset in this notebook

Run the following to create an text file named _MexicoCityNisarDT113.txt_ with the following few lines in it:

In [ ]:
CONFIG_TXT = '''# vim: set filetype=cfg:
mintpy.load.processor      = nisar  #[isce, aria, hyp3, gmtsar, snap, gamma, roipac, nisar], auto for isce
#---------interferogram datasets:
mintpy.load.unwFile        = ../interferograms/NISAR*.h5
mintpy.load.corFile        = auto
mintpy.load.connCompFile   = auto
#---------geometry datasets:
mintpy.load.demFile        = ../DEM/elevation.dem
mintpy.load.incAngleFile   = auto
mintpy.load.azAngleFile    = auto
mintpy.load.waterMaskFile  = auto

mintpy.reference.lalo           = 19.26, -98.88
mintpy.troposphericDelay.method = opera
mintpy.deramp                   = no
mintpy.topographicResidual      = no
mintpy.ionosphericDelay.method  = split_spectrum

# options to speedup the processing (fast but not the best)
mintpy.networkInversion.weightFunc           = no
mintpy.topographicResidual.pixelwiseGeometry = no
'''

write_config_file(config_file, CONFIG_TXT, mode='w')

<div class="alert alert-warning">
<b>Notes:</b> 
The input of MintPy is a stack of interferograms. The multiple pairs of interferograms produced by topsApp.py is NOT a stack.
</div>

![](docs/ifgram_stack.jpg)

Check **more example datasets** from various InSAR processors at:
+ https://mintpy.readthedocs.io/en/latest/demo_dataset/

Check **more example file structures and template setups** from various InSAR processors at:
+ https://mintpy.readthedocs.io/en/latest/dir_structure/

# 2. Time series analysis - small baseline approach

## 2.1 Load the NISAR data into MintPy

MintPy is most consistent with the ISCE direct outputs. However, it supports interferograms processed with other InSAR software including Gamma and SNAP. In this tutorial we are using the NISAR GUNW products.

In [ ]:
!smallbaselineApp.py MexicoCityNisarDT113.txt --dostep load_data
# manually add mintpy.load.processor, as the pre-staged dataset is already loaded into mintpy, causing load_data step fail
ut.add_attribute('inputs/ifgramStack.h5', {'mintpy.load.processor':'nisar'})

By running this command, the "inputs" directory inside the working directory is created and two HDF5 files are produced as

In [ ]:
ls -l inputs

+ `ifgramStack.h5:` this file contains 6 dataset cubes and multiple metadata.

```
unwrapPhase      - 3D array in size of (m, l, w) for unwrapped interferometric phases data cube in radians
coherence        - 3D array in size of (m, l, w) for spatial coherence                data cube
connectComponent - 3D array in size of (m, l, w) for connected commponents            data cube
date             - 2D array in size of (m, 2) in YYYYMMDD format for the reference / secondary dates.
bperp            - 1D array in size of (m,) in meters for perpendicular baselines (average value)
dropIfgram       - 1D array in size of (m,) in boolean to indicate whether an interferogram is used for inversion or ignored
```

where `m` is the number of interferograms, `l` is the number of lines and `w` is the number of columns.

+ `geometryGeo.h5:` this file contains geometrical datasets including height, incidence angle, azimuth angle, shadow layover mask, slant range distance and/or water mask. 

Check more detailed description of the data structure [here](https://mintpy.readthedocs.io/en/latest/api/data_structure/).

<div class="alert alert-info">
<b>info.py:</b> 
To get general infomation about a MintPy product, run info.py on the file. Similar to "gdalinfo".   
</div>

In [ ]:
# !gdalinfo inputs/geometryGeo.h5
!info.py inputs/geometryGeo.h5

In [ ]:
!info.py inputs/ifgramStack.h5

In [ ]:
!info.py inputs/ifgramStack.h5 --date --num --compact

## 2.2 Plot the interferogram network

Before inversion, it's useful to take a look at the network of interferograms. Running `plot_network.py` gives an overview of the network and the average coherence of the stack. 

In [ ]:
plot_network.main('inputs/ifgramStack.h5 -t smallbaselineApp.cfg --figsize 12 4'.split())

Note that with the `--nodisplay` argument, the plots won't be displayed but saved as pdf files in the current directory. Running this command creates multiple files as follows:

+ `coherenceSpatialAvg.txt`: A simple text file that provides an overview to the stack and contains the interferogram dates, average coherence temporal and spatial baseline separation.
+ `coherenceMatrix.pdf`: Shows the avergae coherence pairs between all available pairs in the stack.
+ `coherenceHistory.pdf`: Shows for each acquisition, the minimum and maximum average coherence among all pairs involing this acquisition. Useful to identify bad acquisitions, which usually have very low value for the maximum coherence. Those acquisitions should be dropped.
+ `network.pdf`: Displays the network of interferograms on time-baseline coordinates, colorcoded by avergae coherence of the interferograms. Circles represent the acquisition dates and lines represent the interferograms. Solid lines are the interferograms used for time-series analysis and dashed line are the interferograms ignored in the time-series analysis. 

## 2.3 Generate average spatial coherence and masks

Before running the time-series inversion, one may want to looks at average coherence in the stack. To create a map of average spatial coherence use `temporal_average.py`:

In [ ]:
!temporal_average.py ./inputs/ifgramStack.h5 -d coherence -o avgSpatialCoh.h5
view.main('avgSpatialCoh.h5 --lalo-label --noverbose'.split())
# equivalent command in terminal: view.py avgSpatialCoh.h5 --noverbose

Also one may optionally extract the water mask from the geometry file to be used later for masking the time-series results:

In [ ]:
!generate_mask.py inputs/geometryGeo.h5 waterMask --nonzero -o waterMask.h5
view.main('waterMask.h5 -c gray --lalo-label --noverbose'.split())

The common connected component mask indicates pixels with valid connected component value in all kept interferograms. This is also used to guide the reference point selection.

In [ ]:
!generate_mask.py  inputs/ifgramStack.h5 --nonzero -o maskConnComp.h5 --update
view.main('maskConnComp.h5 -c gray --lalo-label --noverbose'.split())

In [ ]:
!generate_mask.py  avgSpatialCoh.h5  -m 0.2 --base waterMask.h5 -o maskSpatialCoh.h5
view.main('maskSpatialCoh.h5 -c gray --lalo-label --noverbose'.split())

## 2.4 Select reference point

The interferometric phase is relative observation by nature. The phases of each unwrapped interferogram are relative with respect to an arbitrary pixel. Therfore we need to reference all interferograms to a common reference pixel.
The step "reference_point" selects a common reference pixel for the stack of interferograms. The default approach of mintpy is to choose a pixel with highest spatial coherence in the stack. Other options include specifying the longitude and latitude of the desired reference pixel or the line and column number of the refence pixel.    

There are some guidelines for selecting a reference point based on the prior knowledge of the study area in the [mintpy.reference.*](https://github.com/insarlab/MintPy/blob/1c8e7a1890e95fc9a411f0170dacce47443dc897/src/mintpy/defaults/smallbaselineApp.cfg#L113-L125) section.

In [ ]:
!smallbaselineApp.py MexicoCityNisarDT113.txt --dostep reference_point

Running the "reference_step" adds additional attributes "REF_X, REF_Y" and "REF_LON, REF_LAT" to the ifgramStack.h5 file. To see the attributes of the file run info.py:

In [ ]:
!info.py inputs/ifgramStack.h5 | egrep 'REF_'

Note that reference_point does not change the actual values of the unwrapped phase dataset. However, MintPy takes into account the phase at the reference point while performing the time-series inversion. 

## 2.5 Invert network of interferograms for time-series

In the next step we invert the network of differential unwrapped interferograms to estimate the time-series of unwrapped phase with respect to a reference acquisition date, which by default is the first acquisition. The estimated time-series is converted to distance change from radar to target and is provided in meters.  

In [ ]:
!smallbaselineApp.py MexicoCityNisarDT113.txt --dostep invert_network
# copy the output temporal coherence file for later-on comparison
if not os.path.isfile('temporalCoherenceRaw.h5'):
    shutil.copy2('temporalCoherence.h5', 'temporalCoherenceRaw.h5')

The main product generated after running inver_network step, is timeseries.h5. To see the general content of the file run info.py

In [ ]:
!info.py timeseries.h5 --compact

The timeseries file contains three datasets, the time-series which is the interferometric range change for each acquisition relative to the reference acquisition, the "date" dataset which contains the acquisition date for each acquisition and the bperp dataset which contains the timeseries of the perpendicular baseline.  

In [ ]:
view.main('timeseries.h5 --wrap --wrap-range -5 5 -c cmy --noaxis'.split())
# equivalent command in terminal: view.py timeseries.h5 --wrap --wrap-range -5 5 --noaxis

Due to strong ionosphere phase impact, the ionosphere delay needs to be corrected from the timeseries. 

In [ ]:
!smallbaselineApp.py MexicoCityNisarDT113.txt --dostep correct_ionosphere

In [ ]:
view.main('ion.h5 --wrap --wrap-range -5 5 -c cmy --noaxis'.split())

In [ ]:
view.main('timeseries_ion.h5 --wrap --wrap-range -5 5 -c cmy --noaxis'.split())

Individual ionospheric phase screens contain banded phase artifacts oriented along the range direction due to mismatched ionosphere filtering. This banding is magnified in interferogram stacks. While this problem will be fixed in future releases, users can mitigate this effect in the pre-calibration data with further low-pass filtering of the provided ionospheric layer.

![](docs/banded_phase_artifacts.png)

Check **more information** from NISAR Data User Guide at:
+ https://nisar-docs.asf.alaska.edu/


In [ ]:
!smallbaselineApp.py MexicoCityNisarDT113.txt --dostep correct_troposphere


In [ ]:
view.main('timeseries_ion_OPERA.h5 --wrap --wrap-range -5 5 -c cmy --noaxis'.split())

<div class="alert alert-warning">
<b>Question:</b> 
Why we invert a network of small baseline interferograms to estimate a single reference time-series, instead of forming the single-reference network of interferograms at first place? 
</div>

## 2.6 Time-series to velocity

The ground deformation caused by many geophysical or anthropogenic processes are linear at first order approximation. Therefore it is common to estimate the rate of the ground deformation which is the slope of linear fit to the time-series. The step "velocity" estimates the rate of the displacement. 

In [ ]:
!smallbaselineApp.py MexicoCityNisarDT113.txt --dostep velocity
# copy the output velocity file for later-on comparison
if not os.path.isfile('velocityRaw.h5'):
    shutil.copy2('velocity.h5', 'velocityRaw.h5')

In [ ]:
%matplotlib widget
opt = f'--dem ./inputs/geometryGeo.h5 --shade-exag 0.05 --dem-nocontour -v -2.5 2.5 --ylabel-rot 90 --figsize 10 8 -c RdBu_r -u cm/month --lalo-label '
view.main(f'velocity.h5 velocity {opt}'.split())

In [ ]:
opt = f'--dem ./inputs/geometryGeo.h5 --shade-exag 0.05 --dem-nocontour -v -2.5 2.5 --ylabel-rot 90 --figsize 10 8 -c RdBu_r -u cm/month --lalo-label --sub-lat 19.15 19.65 --sub-lon -99.3 -98.75 --lalo-step 0.1 -m maskSpatialCoh.h5'
view.main(f'velocity.h5 velocity {opt}'.split())

<div class="alert alert-info">
<b>Note:</b> 
Negative values indicates that target is moving away from the radar (i.e., subsidence in case of vertical deformation).
Positive values indicates that target is moving towards the radar (i.e., uplift in case of vertical deformation).
</div>

Obvious features in the estimated velocity map:

1) The dominant signal is **rapid land subsidence** over Mexico City, driven by long-term groundwater extraction and compaction of clay-rich lacustrine sediments of the former Lake Texcoco basin (Cabral-Cano et al., 2008; Osmanoğlu et al., 2011; Chaussard et al., 2021).

2) Subsidence is **spatially uneven**: faster rates typically concentrate toward the central–eastern lacustrine plain, while western volcanic highlands / piedmont areas are comparatively stable. The bowl-shaped pattern is a classic Mexico City InSAR signature.

3) For this short NISAR time span, LOS rates are on the order of **cm/month** (roughly consistent with historically reported ~20–40+ cm/year peaks in earlier studies, depending on location and period). Negative LOS values (blue, if using the default MintPy convention) indicate motion **away from the satellite**, i.e. primarily subsidence for near-vertical deformation.

4) Sharp boundaries / high spatial gradients of the subsiding area often follow geotechnical contacts and pre-existing structures; differential settlement in these transition zones is a major infrastructure hazard (Cabral-Cano et al., 2008; Chaussard et al., 2021).

5) The black square marks the **reference pixel**. Because InSAR velocities are relative, choose a coherent, less-deforming location (commonly on more stable ground outside the main lacustrine bowl) when interpreting absolute rates.

References:
+ Cabral-Cano, E., et al. (2008), Space geodetic imaging of rapid ground subsidence in Mexico City, *GSA Bulletin*. https://doi.org/10.1130/B26001.1
+ Osmanoğlu, B., et al. (2011), Mexico City subsidence observed with persistent scatterer InSAR, *ISPRS J. Photogramm. Remote Sens*. https://doi.org/10.1016/j.isprsjprs.2011.02.006
+ Chaussard, E., et al. (2014), Land subsidence in central Mexico detected by ALOS InSAR time-series, *Remote Sensing of Environment*. https://doi.org/10.1016/j.rse.2013.08.038
+ Chaussard, E., et al. (2021), Over a century of sinking in Mexico City..., *JGR Solid Earth*, e2020JB020648. https://doi.org/10.1029/2020JB020648

The estimated velocity also comes with an expression of unecrtainty which is simply based on the goodness of fit while fitting a linear model to the time-series. This quantity is saved in "velocity.h5" under the velocityStd dataset. 

In [ ]:
%matplotlib inline
view.main('velocity.h5 velocityStd -u cm/mon -v 0 1.5 --lalo-label -c RdBu_r'.split())

The estimated standard deviation only represents the goodness of fit and can be biased or maybe under-estimating the actual uncertainty of the product. However, the spatial pattern of the estimated standard deviation is interesting and clearly shows the spatial correltion of noise in the time-series. The uncertainty is distance dependent and increases with increasing distance between pixels. This map shows the uncertainty for each pixle relative to the reference pixel. 

<div class="alert alert-warning">
<b>Question:</b> 
What are the sources of errors that can potentially increase the uncertainty or bias the estimated velocity at this stage?  
</div>